# Marketing Mix Modeling: From Exploration to a Budget Scenario

## 1. Business question and scope
How do weekly search, social, and TV spend relate to revenue after accounting for seasonality, trend, and promotions? Can a model support a small, testable budget reallocation?

This **synthetic educational case study** extends the repository's marketing-funnel EDA. It is not a client result or evidence of real incremental revenue. We generate 156 weeks for a fictional retailer, audit the data, compare a controls-only baseline with a transformed-media regression, and evaluate an unchanged-budget scenario.

**Run:** Python 3.12; install `numpy pandas matplotlib ipykernel`, open this notebook in Jupyter or VS Code, and run all cells in order. No downloads, credentials, or external datasets are needed. Random seeds are fixed; package versions are printed below.

| Field | Meaning | Unit |
|---|---|---|
| week | One observation per Monday | Weekly date |
| search, social, tv | Paid media expenditure | USD thousands per week |
| promo | Scheduled promotion indicator | 0 or 1 |
| revenue | Total retailer revenue | USD thousands per week |

The final 26 weeks are held out. All model choices use earlier weeks only.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({'figure.figsize': (11, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.2})
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, 'pandas:', pd.__version__, 'matplotlib:', matplotlib.__version__)
CHANNELS = ['search', 'social', 'tv']
SEED = 42

## 2. Generate reproducible weekly data
Media has delayed effects and diminishing returns. Our simulator uses channel-specific decay, a Hill curve with slope one, a seasonal baseline, trend, and promotions. The simulator's true parameters are used only to generate data, never for model selection. Model assumptions deliberately simplify the simulator by using one shared decay rate.

The starting adstock is zero: assume no advertising before the observation window. In a real project, load pre-period spend to initialize carryover.

In [ ]:
def adstock(x, decay):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    state = np.zeros(x.shape[1])
    for i, row in enumerate(x):
        state = row + decay * state
        out[i] = state
    return out

def hill(a, half):
    return a / (a + half)

rng = np.random.default_rng(SEED)
n = 156
t = np.arange(n)
spend = rng.lognormal(np.log([18, 14, 22]), [0.40, 0.45, 0.50], size=(n, 3))
promo = (rng.random(n) < 0.20).astype(int)
true_media = hill(adstock(spend, np.array([0.25, 0.50, 0.70])), np.array([28, 35, 70]))
revenue = (110 + 0.18*t + 15*np.sin(2*np.pi*t/52) + 8*np.cos(2*np.pi*t/52)
           + 20*promo + true_media @ np.array([75, 55, 85]) + rng.normal(0, 3, n))
df = pd.DataFrame(spend, columns=CHANNELS)
df.insert(0, 'week', pd.date_range('2023-01-02', periods=n, freq='W-MON'))
df['promo'] = promo
df['revenue'] = revenue
display(df.head())

## 3. Audit before modeling
Confirm the weekly grain, completeness, and valid ranges. A zero spend is valid; missing spend is unknown and should not automatically become zero. Investigate extreme observations before deleting them. This clean simulator intentionally contains no ingestion defects.

In [ ]:
assert df['week'].is_unique and df['week'].is_monotonic_increasing
assert df['week'].diff().dropna().eq(pd.Timedelta(days=7)).all()
assert not df.isna().any().any()
assert np.isfinite(df[CHANNELS + ['revenue']]).all().all()
assert df[CHANNELS + ['revenue']].ge(0).all().all()
assert df['promo'].isin([0, 1]).all()
display(df[CHANNELS + ['revenue']].describe().round(2))
print('Audit passed:', len(df), 'complete, unique weeks.')

## 4. Explore trends, variation, and confounding
Media channels need independent spend variation to distinguish their effects. Correlation with revenue alone cannot separate advertising from demand, promotions, or simultaneous campaigns. On real data, inspect price, distribution, competitor activity, tracking changes, and demand controls before modeling.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(df.week, df.revenue, color='#173f5f')
axes[0].set(title='Synthetic weekly revenue', ylabel='Revenue (USD thousands)')
for channel in CHANNELS:
    axes[1].plot(df.week, df[channel], label=channel, alpha=0.8)
axes[1].set(ylabel='Spend (USD thousands)', xlabel='Week')
axes[1].legend()
plt.tight_layout()
plt.show()
display(df[CHANNELS + ['promo', 'revenue']].corr().round(2))

## 5. Transform spend and define models
We use the unnormalized recursion `a[t] = spend[t] + decay * a[t-1]`, followed by `h(a) = a / (a + half)`. Half-saturation is a multiplier of each channel's **training-period** median adstock. This differs from normalized, finite-lag adstock implementations.

The regression includes an intercept, linear trend, annual sine/cosine terms, and promotion. Ordinary least squares keeps the example transparent; coefficients are unconstrained, so negative media estimates would invalidate positive-return planning here.

Carryover is calculated forward through time, retaining training history in validation and test. No revenue, future spend, or test-period scaling enters a training fit. Evaluation assumes actual spend and promotion are known, so this is conditional prediction rather than an unconditional revenue forecast.

Reference: [Google Meridian: media saturation and lagging](https://developers.google.com/meridian/docs/advanced-modeling/media-saturation-lagging). This notebook is a simplified NumPy implementation, not a Meridian model.

In [ ]:
controls = np.column_stack([np.ones(n), t/52, np.sin(2*np.pi*t/52),
                            np.cos(2*np.pi*t/52), promo])
y = df.revenue.to_numpy()
TEST_START = 130

def design(decay, multiplier, fit_end):
    a = adstock(spend, decay)
    half = np.maximum(np.median(a[:fit_end], axis=0) * multiplier, 1e-8)
    media = hill(a, half)
    return np.column_stack([controls, media]), half

def fit(X, target):
    return np.linalg.lstsq(X, target, rcond=None)[0]

def rmse(actual, pred):
    return float(np.sqrt(np.mean((actual-pred)**2)))

# Mathematical checks for carryover and saturation.
assert np.allclose(adstock(np.array([[10.], [0.], [0.]]), 0.5).ravel(), [10, 5, 2.5])
assert np.allclose(hill(np.array([0., 20.]), 20), [0, 0.5])
assert np.all(np.diff(hill(np.arange(100), 20)) >= 0)

## 6. Select transformations with rolling validation
Three expanding training windows predict the next 13 weeks: 91–103, 104–116, and 117–129 (zero-based). Each fold recomputes its own half-saturation scale. Choose the lowest mean validation RMSE from the declared grid, then refit on weeks 0–129. The final holdout remains untouched until the next section.

In [ ]:
records = []
for decay in [0.0, 0.25, 0.50, 0.75]:
    for multiplier in [0.5, 1.0, 2.0]:
        errors = []
        for end in [91, 104, 117]:
            X_fold, _ = design(decay, multiplier, end)
            beta_fold = fit(X_fold[:end], y[:end])
            errors.append(rmse(y[end:end+13], X_fold[end:end+13] @ beta_fold))
        records.append({'decay': decay, 'multiplier': multiplier,
                        'validation_rmse': np.mean(errors)})
ranking = pd.DataFrame(records).sort_values('validation_rmse').reset_index(drop=True)
display(ranking.head(6).round(3))
best = ranking.iloc[0]
X, half = design(float(best.decay), float(best.multiplier), TEST_START)
beta = fit(X[:TEST_START], y[:TEST_START])
baseline_beta = fit(controls[:TEST_START], y[:TEST_START])
prediction = X @ beta
baseline_prediction = controls @ baseline_beta

## 7. Evaluate once on the final 26 weeks
Compare the media model with the same controls-only baseline. Report RMSE, MAE, and WAPE (sum of absolute errors divided by total actual revenue). A good predictive fit does not establish causal channel attribution. Check residual drift and autocorrelation before trusting uncertainty estimates.

In [ ]:
test = slice(TEST_START, n)
def metrics(pred):
    error = y[test] - pred[test]
    return {'RMSE (USD k)': rmse(y[test], pred[test]),
            'MAE (USD k)': np.mean(np.abs(error)),
            'WAPE (%)': 100*np.sum(np.abs(error))/np.sum(np.abs(y[test]))}
scores = pd.DataFrame({'Controls only': metrics(baseline_prediction),
                       'Media model': metrics(prediction)}).T
display(scores.round(3))
residual = y[test] - prediction[test]
print('Holdout residual lag-1 correlation:', round(pd.Series(residual).autocorr(), 3))
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(df.week[test], y[test], label='Actual', color='#173f5f')
axes[0].plot(df.week[test], prediction[test], label='Media model', color='#ed553b')
axes[0].plot(df.week[test], baseline_prediction[test], label='Controls only', linestyle='--')
axes[0].set(ylabel='Revenue (USD thousands)', title='Chronological holdout')
axes[0].legend()
axes[1].axhline(0, color='gray', linewidth=1)
axes[1].plot(df.week[test], residual)
axes[1].set(ylabel='Residual (USD thousands)', xlabel='Week')
plt.tight_layout()
plt.show()

## 8. Inspect model-implied contributions
With additive features and `h(0) = 0`, the media term is each channel's model-implied contribution relative to zero historical spend, holding controls fixed. Dividing by observed spend gives modeled revenue ROAS, not profit ROI or experimentally measured incrementality. End-of-window carryover is not credited beyond the window.

Keep the controls component and unexplained residual visible; total observed revenue must reconcile. OLS does not guarantee positive contributions, and estimates depend strongly on the assumed response shape.

In [ ]:
media_contribution = X[:, -3:] * beta[-3:]
control_contribution = controls @ beta[:5]
assert np.allclose(control_contribution + media_contribution.sum(axis=1), prediction)
assert np.allclose(control_contribution + media_contribution.sum(axis=1) + (y-prediction), y)
attribution = pd.DataFrame({'Channel': CHANNELS,
    'Holdout spend (USD k)': spend[test].sum(axis=0),
    'Modeled contribution (USD k)': media_contribution[test].sum(axis=0)})
attribution['Modeled revenue ROAS'] = (attribution['Modeled contribution (USD k)'] /
                                        attribution['Holdout spend (USD k)'])
display(attribution.round(3))
print('Holdout controls component (USD k):', round(control_contribution[test].sum(), 2))
print('Holdout unexplained revenue (USD k):', round((y-prediction)[test].sum(), 2))

## 9. Same-budget scenario and response curves
Use the mean spend of the final 13 weeks as the reference allocation. Compare constant weekly budgets at **steady state**, where adstock equals `spend / (1-decay)`. This ignores the transition path and is not a next-week forecast.

Search a coarse grid with each channel constrained to ±20% of reference spend, keeping total budget fixed. Only the selected model is used; these constraints limit changes but do not make the result causal. Curves show modeled weekly media contribution; uncertainty in shape and coefficients is not quantified.

In [ ]:
reference = spend[-13:].mean(axis=0)
budget = reference.sum()
decay = float(best.decay)
def steady_contribution(allocation):
    return hill(np.asarray(allocation)/(1-decay), half) * beta[-3:]

if np.any(beta[-3:] <= 0):
    raise ValueError('Non-positive media coefficients: investigate before budget planning.')
candidates = [reference.copy()]
for search in np.linspace(0.8*reference[0], 1.2*reference[0], 41):
    for social in np.linspace(0.8*reference[1], 1.2*reference[1], 41):
        tv = budget-search-social
        if 0.8*reference[2] <= tv <= 1.2*reference[2]:
            candidates.append(np.array([search, social, tv]))
candidates = np.array(candidates)
values = np.array([steady_contribution(a).sum() for a in candidates])
allocation = candidates[np.argmax(values)]
assert np.isclose(allocation.sum(), budget)
assert np.all(allocation >= 0.8*reference-1e-8)
assert np.all(allocation <= 1.2*reference+1e-8)
gain = steady_contribution(allocation).sum()-steady_contribution(reference).sum()
display(pd.DataFrame({'Channel': CHANNELS, 'Reference weekly spend (USD k)': reference,
                      'Scenario weekly spend (USD k)': allocation,
                      'Change (%)': 100*(allocation/reference-1)}).round(2))
print(f'Model-implied steady-state weekly revenue difference: USD {gain*1000:,.0f}')
print('This is an illustrative model scenario, not an observed business outcome.')
fig, ax = plt.subplots()
for j, channel in enumerate(CHANNELS):
    grid = np.linspace(0, 1.5*reference[j], 100)
    response = hill(grid/(1-decay), half[j])*beta[-3:][j]
    ax.plot(grid, response, label=channel)
    ax.scatter(reference[j], steady_contribution(reference)[j], s=45)
ax.set(xlabel='Weekly spend (USD thousands)', ylabel='Modeled weekly revenue (USD thousands)',
       title='Steady-state response curves; dots mark reference spend')
ax.legend()
plt.tight_layout()
plt.show()

## Results from the seeded run

The media model's holdout RMSE is **4.390 thousand USD**, compared with **10.131 thousand USD** for controls alone; WAPE is **1.534%** versus **3.307%**. Residual lag-1 correlation is **0.397**, suggesting remaining temporal structure that merits investigation.

The selected shared decay is **0.50**, with half-saturation multiplier **1.0**. The constrained scenario increases search spend by **20%** and social by **18%**, while reducing TV by approximately **19.92%**. Its modeled steady-state revenue difference is approximately **3,301 USD per week**. Search reaches its upper bound, so this is a hypothesis for a controlled test, not a validated allocation recommendation. Values may vary slightly by numerical-library version.

These are synthetic, model-dependent results; no realized business lift is claimed.

## 10. Decision, limitations, and next steps
Use the printed holdout scores to judge whether media features improve prediction over seasonality, trend, and promotions. Use the allocation table to formulate a **small experiment**, not to claim realized lift. Average ROAS describes the historical mix; marginal response governs the next dollar.

- **Synthetic truth is unusually favorable:** independent spend variation and fully observed baseline drivers make identification easier than in real marketing data.
- **Structural uncertainty:** shared decay, fixed Hill slope, a small tuning grid, and unconstrained OLS omit channel interactions, coefficient drift, and alternative causal structures. No confidence or credible intervals are supplied.
- **Confounding:** real budgets respond to expected demand. Build a causal graph, justify pre-treatment controls, and calibrate with geo or holdout experiments before interpreting contributions as incremental.
- **Scenario limits:** the grid result is approximate, steady-state, revenue-based, and conditional on the fitted model. It excludes margins, transition carryover, capacity, and costs beyond media. Boundary solutions require closer review.
- **Production adaptation:** replace the simulator with audited weekly data; retain pre-period history; model price, distribution, and demand; test alternate transformations with rolling validation; quantify uncertainty and stability; then pre-register a controlled budget test with incremental profit as the decision metric.

**Deliverable:** a reproducible path from exploratory patterns to an explicitly conditional modeling hypothesis, consistent with this repository's distinction between observation and causality.